# Search Ranking A/B Test: Improving Conversion Without Increasing Booking Time

This project evaluates whether an updated search-ranking experience improves booking conversion on an online travel platform without increasing the time customers need to complete a booking.

The analysis combines session-level activity data with user-level experiment assignments, checks the quality of the randomization, measures the experiment's impact on the primary and guardrail metrics, and concludes with a rollout recommendation.

## Project goals

The analysis is designed to answer three questions:

1. **Experiment integrity:** were users assigned to the control and variant groups in the expected proportions?
2. **Conversion impact:** did the new ranking experience increase the share of sessions that resulted in a booking?
3. **Booking experience:** did the variant increase the time taken to complete a booking?

The final recommendation uses the statistical results together with the direction and size of the observed effects.

## Data overview

The project uses two files that describe the same experiment at different levels.

### `sessions_data.csv` — session-level activity

| column | description |
|--------|-------------|
| `session_id` | Unique session identifier |
| `user_id` | User identifier; missing for anonymous sessions. A user may have multiple sessions. |
| `session_start_timestamp` | When a session started |
| `booking_timestamp` | When a booking was made; missing if no booking was made |
| `time_to_booking` | Minutes from session start to booking; missing if no booking was made |

### `users_data.csv` — experiment assignment

| column | description |
|--------|-------------|
| `user_id` | Unique logged-in user identifier |
| `experiment_group` | Randomized assignment: `control` or `variant` |

The primary outcome, `conversion`, is defined at session level: **1 when `booking_timestamp` is present and 0 otherwise**.

**File setup:** keep both CSV files in the same folder as this notebook before running the analysis.

## Experiment design and evaluation framework

Experiment assignment is recorded at the user level, while booking outcomes are recorded at the session level. A user can therefore contribute multiple sessions to the analysis. Anonymous sessions have no user-level assignment and cannot be attributed to either experiment group.

| Component | Evaluation |
|---|---|
| Randomization check | Chi-square goodness-of-fit test against an expected 50/50 user split |
| Primary metric | Session conversion rate; two-proportion z-test |
| Guardrail metric | Mean time to booking among booked sessions; Welch's two-sample t-test |
| Significance threshold | 10% (`alpha = 0.10`, equivalent to a 90% confidence level) |

### Rollout criteria

- **Primary metric:** the variant must have a positive conversion effect that is statistically significant (`p < alpha`).
- **Guardrail:** the variant must not show a statistically significant increase in time to booking. A statistically significant decrease also satisfies this criterion.

The guardrail rule treats a statistically insignificant difference as acceptable. It is not a formal non-inferiority test and does not prove that the two booking times are equivalent.

In [14]:
import numpy as np
import pandas as pd
from scipy.stats import chisquare, ttest_ind
from statsmodels.stats.proportion import proportions_ztest

pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

In [15]:
confidence_level = 0.90
alpha = 1 - confidence_level
print(f"Confidence level: {confidence_level:.0%} | alpha: {alpha:.2f}")

Confidence level: 90% | alpha: 0.10


In [16]:
sessions = pd.read_csv("sessions_data.csv")
users = pd.read_csv("users_data.csv")

print(f"Sessions: {sessions.shape[0]:,} rows | {sessions.shape[1]} columns")
print(f"Users:    {users.shape[0]:,} rows | {users.shape[1]} columns")
print("Experiment assignment counts:")
display(users["experiment_group"].value_counts().reindex(["control", "variant"]))

Sessions: 16,981 rows | 5 columns
Users:    10,000 rows | 2 columns
Experiment assignment counts:


experiment_group
control    4991
variant    5009
Name: count, dtype: int64

In [17]:
sessions.sample(5, random_state=42)

,session_id,user_id,session_start_timestamp,booking_timestamp,time_to_booking
13205,0j7nwHEgQns1hotv,NOhCZ9DQ8pfEbBjG,2025-01-26 14:26:18.392443180,NaN,NaN
6393,cq7UGnYQj7qCWuWm,cjNZf9eo4Y7b4Og2,2025-01-24 18:38:32.778383255,NaN,NaN
608,mIS2R0e2e1hRrON8,wePd1cdzthHieEUI,2025-01-09 12:27:57.985119104,NaN,NaN
4745,SZpwZ4IWn2o5GGD1,D2NjrHjm467XCjog,2025-01-29 02:07:43.462536573,NaN,NaN
12036,45bYlYJJROrexe32,W2FRmcZQYMaCSrRA,2025-01-03 10:20:12.845188141,NaN,NaN


In [18]:
users.sample(5, random_state=42)

,user_id,experiment_group
6252,VrHt5GggwVdwqRZN,control
4684,LS0BAcXVbu0GwunJ,variant
1731,BKMBCZvmqycz2ejF,variant
4742,HyRTE9GvZfKO8Ccw,variant
4521,vzOKEnIYe9XMPWIX,variant


## 1. Prepare the analysis dataset

Join session records to the user-level experiment assignment. A left join preserves every session, including anonymous sessions. Sessions without an assignment remain in the merged dataset for transparency, but they are excluded from the control-versus-variant comparison.

The `many_to_one` validation ensures that each user has no more than one experiment assignment.

In [19]:
# Basic data-integrity checks
assert sessions["session_id"].is_unique, "session_id should be unique in sessions_data.csv"
assert users["user_id"].is_unique, "user_id should be unique in users_data.csv"
assert set(users["experiment_group"].dropna().unique()).issubset({"control", "variant"})

# Join session-level outcomes to the user-level experiment assignment.
sessions_x_users = sessions.merge(
    users,
    on="user_id",
    how="left",
    validate="many_to_one"
)

# Session conversion: 1 when a booking timestamp is present, otherwise 0.
sessions_x_users["conversion"] = sessions_x_users["booking_timestamp"].notna().astype(int)

# Keep track of unassigned sessions and define the experiment analysis sample.
n_anonymous_sessions = sessions_x_users["experiment_group"].isna().sum()
analysis_data = sessions_x_users.loc[
    sessions_x_users["experiment_group"].isin(["control", "variant"])
].copy()

print(f"Rows in merged dataframe: {len(sessions_x_users):,}")
print(f"Sessions with no experiment assignment (anonymous): {n_anonymous_sessions:,}")
print(f"Assigned sessions used in the A/B analysis: {len(analysis_data):,}")
print("\nConversion definition and counts:")
display(
    analysis_data.groupby("experiment_group")["conversion"]
    .agg(sessions="size", bookings="sum", conversion_rate="mean")
    .reindex(["control", "variant"])
)

Rows in merged dataframe: 16,981
Sessions with no experiment assignment (anonymous): 1,698
Assigned sessions used in the A/B analysis: 15,283

Conversion definition and counts:


,sessions,bookings,conversion_rate
experiment_group,,,
control,7630,1215,0.1592
variant,7653,1392,0.1819


## 2. Check the experiment split (sample ratio mismatch)

Randomization occurs at the **user level**, so the sample ratio mismatch (SRM) check uses the assignment counts in `users_data.csv`, not the number of sessions.

The expected allocation is 50% control and 50% variant. A p-value below `alpha` indicates that the observed split differs significantly from this expected allocation and should be investigated before interpreting the experiment results.

In [20]:
group_order = ["control", "variant"]
observed_user_counts = (
    users["experiment_group"].value_counts()
    .reindex(group_order, fill_value=0)
    .astype(int)
)
expected_user_counts = np.repeat(len(users) / 2, 2)

srm_chi2_stat, srm_chi2_pval = chisquare(
    f_obs=observed_user_counts.to_numpy(),
    f_exp=expected_user_counts
)

srm_results = pd.DataFrame({
    "observed_users": observed_user_counts,
    "expected_users": expected_user_counts
})
display(srm_results)
print(f"SRM chi-square statistic: {srm_chi2_stat:.4f}")
print(f"SRM p-value: {srm_chi2_pval:.6f}")
print("SRM check:", "PASS — no evidence of sample ratio mismatch" if srm_chi2_pval >= alpha else "FAIL — investigate randomization/sample assignment")

,observed_users,expected_users
experiment_group,,
control,4991,"5,000.0000"
variant,5009,"5,000.0000"


SRM chi-square statistic: 0.0324
SRM p-value: 0.857153
SRM check: PASS — no evidence of sample ratio mismatch


## 3. Evaluate the primary metric: conversion

Compare the proportion of sessions that resulted in a booking in the control and variant groups using a two-proportion z-test.

The test is two-sided. The p-value assesses whether the conversion rates differ; the sign of the observed effect is checked separately when applying the rollout criteria.

In [21]:
control = analysis_data.loc[analysis_data["experiment_group"] == "control"]
variant = analysis_data.loc[analysis_data["experiment_group"] == "variant"]

control_conversions = int(control["conversion"].sum())
variant_conversions = int(variant["conversion"].sum())
control_sessions = len(control)
variant_sessions = len(variant)

control_conversion_rate = control["conversion"].mean()
variant_conversion_rate = variant["conversion"].mean()

z_stat_primary, pval_primary = proportions_ztest(
    count=[control_conversions, variant_conversions],
    nobs=[control_sessions, variant_sessions],
    alternative="two-sided"
)

primary_results = pd.DataFrame({
    "sessions": [control_sessions, variant_sessions],
    "bookings": [control_conversions, variant_conversions],
    "conversion_rate": [control_conversion_rate, variant_conversion_rate]
}, index=group_order)
display(primary_results)
print(f"Two-proportion z statistic: {z_stat_primary:.4f}")
print(f"Primary metric p-value: {pval_primary:.8f}")

,sessions,bookings,conversion_rate
control,7630,1215,0.1592
variant,7653,1392,0.1819


Two-proportion z statistic: -3.7220
Primary metric p-value: 0.00019762


## 4. Evaluate the guardrail: time to booking

`time_to_booking` is observed only for sessions that resulted in a booking. Compare the mean booking time among booked sessions using Welch's independent two-sample t-test, which does not assume equal variances between groups.

A negative variant-versus-control effect indicates that the variant is associated with a shorter average time to booking. This metric describes booking time among completed bookings, rather than time-to-event for every session.

In [22]:
control_booking_times = control["time_to_booking"].dropna()
variant_booking_times = variant["time_to_booking"].dropna()

# Welch's two-sample t-test (equal_var=False).
t_stat_guardrail, pval_guardrail = ttest_ind(
    control_booking_times,
    variant_booking_times,
    equal_var=False,
    alternative="two-sided"
)

control_time_mean = control_booking_times.mean()
variant_time_mean = variant_booking_times.mean()

guardrail_results = pd.DataFrame({
    "booked_sessions": [len(control_booking_times), len(variant_booking_times)],
    "mean_time_to_booking_minutes": [control_time_mean, variant_time_mean],
    "std_time_to_booking_minutes": [control_booking_times.std(), variant_booking_times.std()]
}, index=group_order)
display(guardrail_results)
print(f"Welch t statistic: {t_stat_guardrail:.4f}")
print(f"Guardrail metric p-value: {pval_guardrail:.8f}")

,booked_sessions,mean_time_to_booking_minutes,std_time_to_booking_minutes
control,1215,15.0124,4.8121
variant,1392,14.8940,4.9508


Welch t statistic: 0.6182
Guardrail metric p-value: 0.53649971


## 5. Quantify the observed effects

For each metric, calculate the relative difference between the variant and control means:

\[
\text{Relative effect} = \frac{\text{Variant mean} - \text{Control mean}}{\text{Control mean}}
\]

For conversion, a positive effect is favourable. For time to booking, a negative effect is favourable because it represents a reduction in booking time.

In [23]:
effect_size_primary = (
    variant_conversion_rate - control_conversion_rate
) / control_conversion_rate

effect_size_guardrail = (
    variant_time_mean - control_time_mean
) / control_time_mean

print(f"Primary relative effect: {effect_size_primary:.4%}")
print(f"Guardrail relative effect: {effect_size_guardrail:.4%}")

Primary relative effect: 14.2236%
Guardrail relative effect: -0.7885%


## 6. Apply the rollout decision rules

The primary metric passes only when the variant's conversion effect is both **positive** and **statistically significant** (`p < alpha`).

The guardrail passes when either:

- the difference in mean time to booking is not statistically significant (`p >= alpha`), or
- the variant produces a decrease in mean time to booking (negative relative effect).

Both criteria must pass for a full rollout recommendation.

In [24]:
primary_pass = (pval_primary < alpha) and (effect_size_primary > 0)
guardrail_pass = (pval_guardrail >= alpha) or (effect_size_guardrail < 0)

decision_full_on = "yes" if (primary_pass and guardrail_pass) else "no"

print(f"Alpha: {alpha:.2f}")
print(f"Primary criterion passed: {primary_pass}")
print(f"Guardrail criterion passed: {guardrail_pass}")
print(f"Full-rollout decision: {decision_full_on}")

Alpha: 0.10
Primary criterion passed: True
Guardrail criterion passed: True
Full-rollout decision: yes


## 7. Results and recommendation

This section brings the experiment checks, p-values, relative effects, and final rollout decision together. Run the notebook from top to bottom to populate the results with values calculated from the CSV files.

In [25]:
required_outputs = pd.Series({
    "srm_chi2_pval": srm_chi2_pval,
    "pval_primary": pval_primary,
    "pval_guardrail": pval_guardrail,
    "effect_size_primary": effect_size_primary,
    "effect_size_guardrail": effect_size_guardrail,
    "decision_full_on": decision_full_on
}, name="value")
display(required_outputs.to_frame())

print("\nInterpretation:")
print(
    f"The variant conversion rate was {variant_conversion_rate:.2%}, "
    f"compared with {control_conversion_rate:.2%} in control "
    f"({effect_size_primary:+.2%} relative change)."
)
print(
    f"Mean time to booking was {variant_time_mean:.2f} minutes in variant "
    f"and {control_time_mean:.2f} minutes in control "
    f"({effect_size_guardrail:+.2%} relative change; p={pval_guardrail:.4f})."
)
if decision_full_on == "yes":
    print("Recommendation: FULL ROLLOUT. The primary metric is significantly improved and the guardrail criterion is satisfied.")
else:
    print("Recommendation: PULL BACK. The full-rollout criteria are not both satisfied.")

,value
srm_chi2_pval,0.8572
pval_primary,0.0002
pval_guardrail,0.5365
effect_size_primary,0.1422
effect_size_guardrail,-0.0079
decision_full_on,yes



Interpretation:
The variant conversion rate was 18.19%, compared with 15.92% in control (+14.22% relative change).
Mean time to booking was 14.89 minutes in variant and 15.01 minutes in control (-0.79% relative change; p=0.5365).
Recommendation: FULL ROLLOUT. The primary metric is significantly improved and the guardrail criterion is satisfied.


## 8. Sensitivity analysis: account for repeated sessions per user

Experiment assignment is at the user level, but the main conversion analysis treats sessions as observations. Users with multiple sessions can therefore contribute repeated observations. This additional sensitivity analysis aggregates each user's session conversion rate and mean observed booking time, so each user contributes one observation to the group comparisons.

The sensitivity analysis is a diagnostic rather than a replacement for the main session-level analysis. The user-level conversion measure is the average of each user's session outcomes, which is not exactly the same estimand as the pooled session conversion rate. Booking time is also compared among users with at least one booked session.

In [26]:
user_metrics = (
    analysis_data
    .groupby(["experiment_group", "user_id"], as_index=False)
    .agg(
        user_conversion_rate=("conversion", "mean"),
        user_mean_time_to_booking=("time_to_booking", "mean"),
        sessions_per_user=("session_id", "size")
    )
)

control_users = user_metrics.loc[user_metrics["experiment_group"] == "control"]
variant_users = user_metrics.loc[user_metrics["experiment_group"] == "variant"]

# Each user contributes exactly one conversion-rate observation.
user_primary_test = ttest_ind(
    control_users["user_conversion_rate"],
    variant_users["user_conversion_rate"],
    equal_var=False,
    alternative="two-sided"
)

# Booking-time comparison among users with at least one booked session.
control_user_times = control_users["user_mean_time_to_booking"].dropna()
variant_user_times = variant_users["user_mean_time_to_booking"].dropna()
user_guardrail_test = ttest_ind(
    control_user_times,
    variant_user_times,
    equal_var=False,
    alternative="two-sided"
)

user_effect_size_primary = (
    variant_users["user_conversion_rate"].mean() - control_users["user_conversion_rate"].mean()
) / control_users["user_conversion_rate"].mean()
user_effect_size_guardrail = (
    variant_user_times.mean() - control_user_times.mean()
) / control_user_times.mean()

user_level_sensitivity = pd.DataFrame({
    "control_mean": [control_users["user_conversion_rate"].mean(), control_user_times.mean()],
    "variant_mean": [variant_users["user_conversion_rate"].mean(), variant_user_times.mean()],
    "variant_relative_effect": [user_effect_size_primary, user_effect_size_guardrail],
    "p_value": [user_primary_test.pvalue, user_guardrail_test.pvalue]
}, index=["user-level session conversion rate", "user-level mean time to booking (minutes)"])
display(user_level_sensitivity)

print("User-level sensitivity supports the same primary/guardrail conclusion:")
print(f"  Conversion: p={user_primary_test.pvalue:.6f}, relative effect={user_effect_size_primary:+.2%}")
print(f"  Time to booking: p={user_guardrail_test.pvalue:.6f}, relative effect={user_effect_size_guardrail:+.2%}")

,control_mean,variant_mean,variant_relative_effect,p_value
user-level session conversion rate,0.1591,0.1796,0.1291,0.0019
user-level mean time to booking (minutes),14.9642,14.9011,-0.0042,0.7454


User-level sensitivity supports the same primary/guardrail conclusion:
  Conversion: p=0.001926, relative effect=+12.91%
  Time to booking: p=0.745434, relative effect=-0.42%
